# Document Chunking for Retrieval

Embeddings work best on **focused spans of text** — a section, a paragraph, a procedure — not on an entire 200-page manual compressed into one vector. **Chunking** splits source documents into pieces that are embedded, stored, and retrieved independently. It is one of the highest-leverage steps in a RAG pipeline: bad chunks cause retrieval misses even when the embedding model and vector DB are perfect.

This notebook explains why chunking matters, the precision–context trade-off, major strategies (fixed-size, recursive separators, structure-aware, semantic), overlap, metadata design, token-based sizing, and live demonstrations showing how chunk choices change retrieval rankings.

Prerequisites: **01–03** in this section. Related: **Tokens and Context Windows** (GenAI Fundamentals), **04. Document Chunking** themes in that notebook.

In [ ]:
import re
import numpy as np

np.set_printoptions(precision=4, suppress=True)

---

## 1. Why Chunk?

### 1.1 The Single-Vector Problem

One embedding per long document averages many topics into one direction in vector space. A query about **migrations** competes with **authentication**, **billing**, and **onboarding** in the same vector — the signal dilutes.

| Approach | Retrieval behavior |
|----------|-------------------|
| **Whole document** | Vague match; right section may not dominate |
| **Focused chunks** | Query aligns with the specific chunk containing the answer |

### 1.2 Drivers for Chunking

| Reason | Explanation |
|--------|-------------|
| **Embedding token limits** | Models cap input length per API call |
| **Retrieval precision** | Smaller units match specific facts |
| **LLM context budget** | RAG sends top-k chunks; each chunk consumes tokens |
| **Citation granularity** | "See Authentication section" vs "see entire handbook" |
| **Ingest cost** | More chunks = more embed API calls — balance quality and cost |

Chunking sits between **raw documents** and **vector indexes** — get it wrong and downstream steps cannot recover.

---

## 2. The Chunk Size Trade-Off

```
Small chunks                    Large chunks
─────────────                   ────────────
+ Precise retrieval             + More local context per hit
+ Fits more distinct hits in k  + Fewer total vectors to manage
− Missing surrounding context   − Mixed topics in one vector
− More vectors / storage        − Lower retrieval precision
```

| Chunk size | Risk |
|------------|------|
| **Too small** (1–2 sentences) | Answer split across chunks; pronouns lose referents |
| **Too large** (multi-page) | Returns irrelevant paragraphs with the answer |
| **Sweet spot** (domain-specific) | Often **300–800 tokens** — **validate on eval set** (notebook 09) |

There is no universal optimum — legal contracts, API docs, and chat logs need different sizes.

---

## 3. Chunking Strategies Overview

| Strategy | Mechanism | Best when |
|----------|-----------|----------|
| **Fixed character/token** | Split every N units | Uniform prose, quick baseline |
| **Recursive separators** | Try `\n\n`, `\n`, `. `, space | General text; avoids mid-word cuts |
| **Sentence-based** | Pack sentences until size limit | Narrative, support articles |
| **Structure-aware** | Split on Markdown/HTML headers | Docs with clear sections |
| **Semantic** | Merge/split by embedding similarity | Highest quality; more compute |
| **Document-native** | PDF pages, slides, table rows | Format-specific pipelines |

**Production path:** start **structure-aware** if headers exist; else **recursive** token splitter; tune size/overlap with evals.

---

## 4. Fixed-Size Chunking with Overlap

The simplest algorithm: window of `chunk_size` characters (or tokens), advance by `chunk_size - overlap`.

```
Chunk 1: [AAAAAAAA|BBBB]
Chunk 2:      [BBBB|CCCCCCCC]
              ^^^^ overlap preserves boundary facts
```

| Parameter | Typical starting point |
|-----------|------------------------|
| `chunk_size` | 500–1000 **tokens** (not chars) in production |
| `overlap` | 10–20% of chunk_size |

**Downside:** ignores document structure — may split `"Alembic manages"` from `"schema migrations"`.

In [ ]:
def chunk_fixed(text: str, size: int = 120, overlap: int = 30) -> list[dict]:
    if overlap >= size:
        raise ValueError("overlap must be < size")
    chunks = []
    start = 0
    while start < len(text):
        end = min(start + size, len(text))
        piece = text[start:end]
        chunks.append({"text": piece, "start": start, "end": end, "char_len": len(piece)})
        if end >= len(text):
            break
        start = end - overlap
    return chunks


sample = "ABCDEFGHIJKLMNOPQRSTUVWXYZ" * 3
for i, c in enumerate(chunk_fixed(sample, size=20, overlap=5)):
    print(f"[{i}] chars {c['char_len']:2d}  pos {c['start']:3d}-{c['end']:3d}  {c['text'][:25]}...")

---

## 5. Recursive Separator Splitting

Try **large separators first**, then subdivide oversize pieces. Common separator priority:

1. `\n\n` (paragraph)
2. `\n` (line)
3. `. ` (sentence)
4. ` ` (word)

This is the idea behind LangChain's `RecursiveCharacterTextSplitter` — we implement a minimal version here without extra dependencies.

In [ ]:
def recursive_split(text: str, chunk_size: int, separators: list[str] | None = None) -> list[str]:
    """Greedy recursive split — keeps pieces <= chunk_size when possible."""
    if separators is None:
        separators = ["\n\n", "\n", ". ", " "]
    if len(text) <= chunk_size:
        return [text.strip()] if text.strip() else []

    sep = separators[0]
    rest = separators[1:]
    parts = text.split(sep) if sep else list(text)

    chunks: list[str] = []
    buf = ""
    for part in parts:
        candidate = (buf + sep + part) if buf else part
        if len(candidate) <= chunk_size:
            buf = candidate
        else:
            if buf:
                chunks.append(buf.strip())
            if len(part) <= chunk_size:
                buf = part
            elif rest:
                chunks.extend(recursive_split(part, chunk_size, rest))
                buf = ""
            else:
                chunks.append(part[:chunk_size].strip())
                buf = part[chunk_size:]
    if buf.strip():
        chunks.append(buf.strip())
    return chunks


demo_text = "First paragraph about auth.\n\nSecond paragraph about Alembic migrations.\n\nThird about FastAPI endpoints."
for i, c in enumerate(recursive_split(demo_text, chunk_size=60)):
    print(f"[{i}] ({len(c)} chars) {c}")

---

## 6. Structure-Aware Chunking (Markdown Headers)

Technical docs often have `#` / `##` headers. Splitting on headers keeps **semantic sections** together — ideal for API and internal wikis.

In [ ]:
DOCUMENT = """# Authentication

Our API uses JWT tokens for stateless authentication. Clients obtain a token
from POST /auth/login with valid credentials. Tokens expire after 15 minutes.

# Database Migrations

Alembic manages schema migrations for SQLAlchemy projects. Run `alembic upgrade head`
to apply pending migrations in CI and production. Revision files live in versions/.

# Notes API

The Notes API stores notes in memory and exposes REST endpoints built with FastAPI.
Use GET /notes to list all notes for the authenticated user.
"""


def chunk_by_markdown_headers(text: str) -> list[dict]:
    pattern = re.compile(r"^(#+ .+)$", re.MULTILINE)
    parts = pattern.split(text)
    chunks = []
    # pattern.split alternates: [preamble, header, body, header, body, ...]
    if parts[0].strip():
        chunks.append({"section": "(preamble)", "text": parts[0].strip()})
    i = 1
    while i < len(parts) - 1:
        header = parts[i].strip()
        body = parts[i + 1].strip()
        chunks.append({"section": header, "text": f"{header}\n\n{body}".strip()})
        i += 2
    return chunks


header_chunks = chunk_by_markdown_headers(DOCUMENT)
for i, c in enumerate(header_chunks):
    print(f"[{i}] {c['section']}")
    print(f"    {c['text'][:90].replace(chr(10), ' ')}...\n")

---

## 7. Token-Based Chunk Sizing

Character counts ≠ token counts. Production splitters should target **tokens** to align with embedding limits and LLM context accounting.

In [ ]:
try:
    import tiktoken

    enc = tiktoken.get_encoding("cl100k_base")

    def token_len(s: str) -> int:
        return len(enc.encode(s))

    lines = [
        "Short line.",
        DOCUMENT[:200],
        "Code: `alembic upgrade head` runs migrations.",
    ]
    for line in lines:
        print(f"tokens={token_len(line):3d}  chars={len(line):3d}  ratio={len(line)/max(token_len(line),1):.1f} chars/token")
except ImportError:
    print("Install tiktoken for token-based sizing demos.")

---

## 8. Metadata per Chunk

Each chunk should carry **provenance** for filtering, debugging, and UI citations.

```python
{
    "chunk_id": "api-docs#migrations#0",
    "source": "api-docs.md",
    "section": "Database Migrations",
    "page": 12,              # PDFs
    "doc_type": "internal",
    "version": "2025-03",
    "char_start": 840,       # optional offsets
}
```

| Field | Purpose |
|-------|---------|
| `source` | File path or URL |
| `section` | Header or chapter title |
| `chunk_id` | Stable unique id for upserts |
| `parent_id` | Link small chunks to parent doc (parent–child pattern) |

Store metadata in the vector DB (notebook 07) — not only in a sidecar file.

---

## 9. Parent–Child and Small-to-Large Retrieval

Advanced pattern:

1. **Index small chunks** for precise retrieval
2. When a small chunk hits, fetch **parent section** (or neighboring chunks) for the LLM

```
Retrieve: small chunk "alembic upgrade head"
Send to LLM: full "Database Migrations" section (parent)
```

Balances **precision** (small index) with **context** (large generation input).

---

## 10. Semantic Chunking (Overview)

**Semantic chunking** embeds sentences (or micro-spans), measures similarity between adjacent units, and splits where similarity **drops** — boundaries between topics.

| Pros | Cons |
|------|------|
| Topic-coherent chunks | Extra embed calls during ingest |
| Adapts to content | Slower, harder to reproduce |

Use when recursive/structure splitters fail evals and budget allows. Libraries: LangChain `SemanticChunker`, custom pipelines.

---

## 11. Pre-Processing Before Chunk

| Step | Why |
|------|-----|
| Normalize whitespace | Avoid empty or tiny chunks |
| Strip boilerplate (nav, footers) | Reduces noise in HTML/PDF extract |
| De-duplicate exact paragraphs | Saves embed cost |
| Preserve code blocks | Don't split mid-fence in code-heavy docs |

Garbage in → garbage chunks → garbage retrieval.

---

## 12. Chunk Count and Ingest Cost

$$\text{chunks} \approx \frac{\text{document tokens}}{\text{chunk\_size} - \text{overlap}}$$

More chunks → more embedding API tokens → higher storage. **Overlap duplicates text** across chunks (intentional cost for better boundary recall).

| Docs | Tokens | ~500-token chunks | Rough embed calls |
|------|--------|-------------------|-------------------|
| 100 pages wiki | 80k | ~160 | 160 |
| 10k support tickets | 2M | ~4k | 4k |

Right-size chunks before worrying about vector DB vendor pricing.

---

## 13. Demonstration: Compare Chunkers Side by Side

In [ ]:
def chunk_by_paragraphs(text: str, max_chars: int = 220) -> list[dict]:
    paragraphs = [p.strip() for p in text.split("\n\n") if p.strip()]
    chunks, buf = [], ""
    for para in paragraphs:
        candidate = f"{buf}\n\n{para}".strip() if buf else para
        if len(candidate) <= max_chars:
            buf = candidate
        else:
            if buf:
                chunks.append({"text": buf, "char_len": len(buf)})
            buf = para
    if buf:
        chunks.append({"text": buf, "char_len": len(buf)})
    return chunks


strategies = {
    "fixed-80": [c["text"] for c in chunk_fixed(DOCUMENT, size=80, overlap=15)],
    "fixed-200": [c["text"] for c in chunk_fixed(DOCUMENT, size=200, overlap=40)],
    "paragraph": [c["text"] for c in chunk_by_paragraphs(DOCUMENT)],
    "markdown": [c["text"] for c in chunk_by_markdown_headers(DOCUMENT)],
    "recursive": recursive_split(DOCUMENT, chunk_size=180),
}

for name, chunks in strategies.items():
    lens = [len(c) for c in chunks]
    print(f"{name:12s}  count={len(chunks):2d}  min/avg/max chars: {min(lens)}/{sum(lens)//len(lens)}/{max(lens)}")

---

## 14. Demonstration: Retrieval Impact (OpenAI Embeddings)

Same document, different chunk strategies — which retrieves best for a migration query?

In [ ]:
OPENAI_API_KEY = "sk-proj-placeholder-replace-with-your-real-key"

from openai import OpenAI

client = OpenAI(api_key=OPENAI_API_KEY)
MODEL = "text-embedding-3-small"


def embed(texts: list[str]) -> np.ndarray:
    r = client.embeddings.create(model=MODEL, input=texts)
    ordered = sorted(r.data, key=lambda x: x.index)
    return np.array([d.embedding for d in ordered])


def best_match(query: str, chunks: list[str]) -> tuple[int, float, str]:
    if not chunks:
        return -1, 0.0, ""
    vecs = embed(chunks)
    q = embed([query])[0]
    scores = (vecs @ q) / (np.linalg.norm(vecs, axis=1) * np.linalg.norm(q))
    idx = int(np.argmax(scores))
    return idx, float(scores[idx]), chunks[idx]


query = "How do I apply database migrations in production?"
print(f"Query: {query}\n")

for name, chunks in strategies.items():
    idx, score, text = best_match(query, chunks)
    preview = text.replace("\n", " ")[:100]
    print(f"{name:12s}  score={score:.4f}  chunk[{idx}]  {preview}...")

Expect **markdown** or **paragraph** strategies to rank the Alembic section highest; **fixed-80** may split critical terms across chunks and score lower.

---

## 15. Demonstration: Overlap Effect

Boundary fact split across chunks without overlap — does overlap recover it?

In [ ]:
BOUNDARY_DOC = (
    "...prefix text about deployment. "
    "CRITICAL: run alembic upgrade head before serving traffic. "
    "Suffix text about monitoring and alerts."
)

query = "alembic upgrade before serving traffic"

for overlap in (0, 30):
    chunks = [c["text"] for c in chunk_fixed(BOUNDARY_DOC, size=70, overlap=overlap)]
    _, score, text = best_match(query, chunks)
    has_critical = "CRITICAL" in text and "alembic" in text
    print(f"overlap={overlap:2d}  chunks={len(chunks)}  score={score:.4f}  full_critical_in_best={has_critical}")
    print(f"         best: {text[:80].replace(chr(10), ' ')}...\n")

---

## 16. Building Chunk Records for Ingest

Production ingest maps each chunk to a record ready for the vector DB:

In [ ]:
def build_chunk_records(
    source: str,
    chunks: list[dict],
) -> list[dict]:
    records = []
    for i, c in enumerate(chunks):
        records.append(
            {
                "id": f"{source}#{i}",
                "text": c["text"],
                "metadata": {
                    "source": source,
                    "section": c.get("section", ""),
                    "chunk_index": i,
                },
            }
        )
    return records


records = build_chunk_records("api-docs.md", chunk_by_markdown_headers(DOCUMENT))
for r in records:
    print(r["id"], "|", r["metadata"]["section"], "|", r["text"][:50].replace("\n", " "), "...")

---

## 17. Tuning Workflow

1. Pick a **baseline** splitter (markdown or recursive, ~500 tokens)
2. Build **labeled queries** with expected sections (notebook 09)
3. Measure **Recall@k** across chunk sizes (256, 512, 1024) and overlaps (0%, 10%, 20%)
4. Promote settings that win without blowing ingest budget
5. **Version** chunk config (`chunk_v3_512_15pct`) alongside embedding model

---

## 18. Common Mistakes

| Mistake | Effect | Fix |
|---------|--------|-----|
| One vector per PDF | Vague retrieval | Chunk by section |
| Chars not tokens | Wrong size vs limits | Use tiktoken |
| Zero overlap on dense facts | Boundary misses | 10–20% overlap |
| No metadata | Can't cite or filter | Attach source/section |
| Split mid code fence | Broken examples | Structure-aware rules |
| Huge chunks + small k | LLM context waste | Smaller chunks or parent–child |

---

## 19. Summary

**Chunking** converts documents into retrieval-sized units. Balance **precision** (small chunks) against **context** (larger chunks). Prefer **structure-aware** or **recursive** splitters over naive fixed windows; add **overlap**; attach **metadata**; size in **tokens**.

Demonstrations compared five strategies, measured retrieval scores on a migration query, and showed overlap recovering boundary facts. Tune with **Recall@k** on real queries — not guesswork.

Next: **05. Introduction to Vector Databases** — store and query chunked embeddings at scale.